# 04c Build Resistance-Only Screened Dataset

Apply the resistance-only Jev run to candidate records. Only explicit resistance findings for the exact pathogen–antibiotic pair enter annual counts. Validated runs support final analysis; the explicitly marked provisional mode supports exploratory analysis before human calibration.

## Inputs

- Candidate pair-year PMID lists from `candidate_amr_pair_year_counts.parquet`.
- Complete System 1 predictions from the configured resistance-only run.
- Human-reviewed calibration for final analysis, or an explicitly configured provisional cutoff for exploratory analysis.

## Outputs

- `yearly_amr_screened_pairs.parquet` and provenance recording whether the cutoff is validated or provisional.

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipeline_config import cfg, cfg_path, load_pipeline_config

from amr.screening import build_yearly_screened_pairs, prepare_yearly_candidate_pmids
from amr.screening.calibration import write_screening_provenance


In [3]:
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
OUTPUT_YEARLY_AMR_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_candidates", PROJECT_ROOT)
OUTPUT_YEARLY_SCREEN_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
OUTPUT_UNIFIED_DECISIONS_PATH = cfg_path(PIPELINE_CONFIG, "paths.unified_decisions", PROJECT_ROOT)
OUTPUT_SYSTEM1_DECISIONS_PATH = cfg_path(PIPELINE_CONFIG, "paths.system1_decisions", PROJECT_ROOT)
ACTIVE_PREDICTION_SOURCE = str(cfg(PIPELINE_CONFIG, "screening.active_source", "llm")).lower()
SYSTEM1_PROBABILITY_THRESHOLD = float(cfg(PIPELINE_CONFIG, "screening.system1_probability_threshold", 0.5))
SYSTEM1_RUN_ID = str(cfg(PIPELINE_CONFIG, "system1.run_id", "jev-default"))
SYSTEM1_QUESTION_VERSION = str(cfg(PIPELINE_CONFIG, "system1.question_version", "unknown"))
SYSTEM1_CALIBRATION_STATUS = str(cfg(PIPELINE_CONFIG, "system1.threshold_calibration_status", "pending"))
SYSTEM1_TARGET_PRECISION = float(cfg(PIPELINE_CONFIG, "system1.target_precision", 0.90))
SCREENED_INCLUDE_DECISIONS = tuple(cfg(PIPELINE_CONFIG, "llm.screened_include_decisions", ["strict", "yes"]))
if ACTIVE_PREDICTION_SOURCE not in {"llm", "system1"}:
    raise ValueError("screening.active_source must be 'llm' or 'system1'.")
if not 0.0 <= SYSTEM1_PROBABILITY_THRESHOLD <= 1.0:
    raise ValueError("screening.system1_probability_threshold must be between 0 and 1.")


## Rebuild canonical screened output from disk

In [4]:
if ACTIVE_PREDICTION_SOURCE != "system1":
    raise RuntimeError("The canonical AMR-SI workflow now requires the validated resistance-only System 1 screen.")
if SYSTEM1_CALIBRATION_STATUS not in {"validated", "provisional_uncalibrated"}:
    raise RuntimeError("The resistance-only Jev cutoff is pending. Calibrate it or explicitly select provisional_uncalibrated for exploratory outputs.")

if not OUTPUT_YEARLY_AMR_PATH.exists():
    raise FileNotFoundError(f"Missing {OUTPUT_YEARLY_AMR_PATH}. Run notebook 04 collection first.")
yearly_amr_df = pd.read_parquet(OUTPUT_YEARLY_AMR_PATH)
if ACTIVE_PREDICTION_SOURCE == "llm":
    prediction_path = OUTPUT_UNIFIED_DECISIONS_PATH
    builder_kwargs = {"include_decisions": SCREENED_INCLUDE_DECISIONS}
else:
    prediction_path = OUTPUT_SYSTEM1_DECISIONS_PATH
    builder_kwargs = {
        "probability_threshold": SYSTEM1_PROBABILITY_THRESHOLD,
        "system1_run_id": SYSTEM1_RUN_ID,
    }
if not prediction_path.exists():
    raise FileNotFoundError(f"Missing selected prediction store: {prediction_path}. Run notebook 04b first.")
prediction_df = pd.read_parquet(prediction_path)
print(f"Prediction source: {ACTIVE_PREDICTION_SOURCE}; rows loaded: {len(prediction_df):,}")
active_predictions = prediction_df.loc[prediction_df["system1_run_id"].astype(str).eq(SYSTEM1_RUN_ID)].copy()
expected = prepare_yearly_candidate_pmids(yearly_amr_df)
key_cols = ["pmid", "pathogen", "antibiotic", "year"]
expected_keys = expected[key_cols].drop_duplicates()
scored_keys = active_predictions.loc[active_predictions["system1_status"].isin(["ok", "no_abstract"]), key_cols].drop_duplicates()
missing_screen = expected_keys.merge(scored_keys, on=key_cols, how="left", indicator=True).query("_merge == 'left_only'")
unresolved = active_predictions.loc[~active_predictions["system1_status"].isin(["ok", "no_abstract"])]
if not missing_screen.empty or not unresolved.empty:
    raise RuntimeError(
        f"Resistance-only run incomplete: {len(missing_screen):,} candidate decisions missing; "
        f"{len(unresolved):,} decisions unresolved. Resume notebook 04b before rebuilding counts."
    )
if ACTIVE_PREDICTION_SOURCE == "system1":
    print(f"System 1 run: {SYSTEM1_RUN_ID}; P(yes) threshold: {SYSTEM1_PROBABILITY_THRESHOLD:.3f}")

yearly_screened_df = build_yearly_screened_pairs(
    yearly_amr_df,
    prediction_df,
    prediction_source=ACTIVE_PREDICTION_SOURCE,
    output_path=str(OUTPUT_YEARLY_SCREEN_PATH),
    merge_with_existing_output=False,
    **builder_kwargs,
)
print(f"Screened yearly rows: {len(yearly_screened_df):,}")
print(f"Unique pairs: {len(yearly_screened_df[['pathogen', 'antibiotic']].drop_duplicates()):,}")
print(f"Screened output replaced at: {OUTPUT_YEARLY_SCREEN_PATH}")
import json
calibration_report_path = PROJECT_ROOT / "outputs" / "review" / "resistance_screen_calibration.json"
calibration_report = json.loads(calibration_report_path.read_text()) if SYSTEM1_CALIBRATION_STATUS == "validated" and calibration_report_path.exists() else {}
if SYSTEM1_CALIBRATION_STATUS == "provisional_uncalibrated":
    print("PROVISIONAL EXPLORATORY OUTPUT: cutoff has not passed human calibration.")
provenance_path = OUTPUT_YEARLY_SCREEN_PATH.with_suffix(".provenance.json")
write_screening_provenance(provenance_path, {
    "screening_source": "system1",
    "run_id": SYSTEM1_RUN_ID,
    "question_version": SYSTEM1_QUESTION_VERSION,
    "probability_threshold": SYSTEM1_PROBABILITY_THRESHOLD,
    "threshold_calibration_status": SYSTEM1_CALIBRATION_STATUS,
    "target_precision": SYSTEM1_TARGET_PRECISION,
    "holdout_precision": calibration_report.get("holdout_precision"),
    "provisional_note": "User-requested exploratory run before human calibration; do not interpret as meeting the precision target." if SYSTEM1_CALIBRATION_STATUS == "provisional_uncalibrated" else None,
})
print(f"Screening provenance written to: {provenance_path}")
yearly_screened_df.head(10)


Prediction source: system1; rows loaded: 529,091
System 1 run: jev-1.13-amr-support-v1; P(yes) threshold: 0.750
Screened yearly rows: 64,721
Unique pairs: 15,539
Screened output replaced at: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/parquets/yearly_amr_screened_pairs.parquet


,pathogen,antibiotic,year,count,pmids
0,Abiotrophia defectiva,Amoxicillin,2000,1,[11109021]
1,Abiotrophia defectiva,Amoxicillin,2018,1,[30514000]
2,Abiotrophia defectiva,Ampicillin,2018,1,[30514000]
3,Abiotrophia defectiva,Ampicillin,2021,1,[34252304]
4,Abiotrophia defectiva,Benzylpenicillin,2019,1,[31049562]
5,Abiotrophia defectiva,Cefaclor,2018,1,[30514000]
6,Abiotrophia defectiva,Cefepime,2016,1,[27216060]
7,Abiotrophia defectiva,Cefotaxime,2016,1,[27216060]
8,Abiotrophia defectiva,Ceftazidime,2018,1,[30514000]
9,Abiotrophia defectiva,Ceftriaxone,2000,1,[11109021]
